## Smart Table Filler

### Auto-fill CSV / Excel / any tabular file using a Frontier or Open-Source LLM

**Pipeline:**
1. Upload any tabular file (CSV, Excel, TSV, JSON, Parquet)
2. Analyse which cells are filled vs. missing
3. Scrape a user-supplied URL for reference context
4. Use a Frontier LLM (OpenAI / Anthropic) or local model (Ollama) to fill the gaps
5. Review and export the completed table via a Gradio UI

### Imports

In [2]:
# Install missing package (makes playwright available in this Jupyter session)
%pip install playwright

import os
import re
import json
import time
import warnings

import pandas as pd
import numpy as np
import requests
import gradio as gr

from dotenv import load_dotenv
from bs4 import BeautifulSoup
from pathlib import Path
from tqdm import tqdm

from openai import OpenAI
from anthropic import Anthropic
# pip install playwright && playwright install chromium
#from playwright.sync_api import sync_playwright

warnings.filterwarnings("ignore")

Note: you may need to restart the kernel to use updated packages.


c:\Users\Nii Quaynor\projects\Caishen\.venv\Scripts\python.exe: No module named pip


### Configuration

In [3]:
load_dotenv(override=True)

# ── Model options ─────────────────────────────────────────────────────────────
# Frontier
OPENAI_MODEL   = "gpt-4o-mini"              # swap to gpt-4o for best quality
ANTHROPIC_MODEL = "claude-sonnet-4-6"       # swap to claude-opus-4-6 for best quality

# Open-source via Ollama (must have Ollama running locally)
OLLAMA_MODEL   = "llama3"                   # swap to any model pulled via `ollama pull`
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Default choice: "openai" | "anthropic" | "ollama"
DEFAULT_PROVIDER = "openai"

# Supported file extensions
SUPPORTED_EXT = [".csv", ".xlsx", ".xls", ".tsv", ".json", ".parquet"]

### LLM Clients

We instantiate clients for each provider. Only the selected provider is called at runtime.

In [4]:
openai_client    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
ollama_client    = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")  # Ollama uses OpenAI-compatible API

### File Loader

Accepts CSV, Excel (.xlsx / .xls), TSV, JSON, and Parquet.
Auto-detects the header row for Excel files with leading blank rows.

In [5]:
LOADERS = {
    ".csv":     lambda f: pd.read_csv(f),
    ".tsv":     lambda f: pd.read_csv(f, sep="\t"),
    ".xlsx":    lambda f: pd.read_excel(f, header=None),
    ".xls":     lambda f: pd.read_excel(f, header=None),
    ".json":    lambda f: pd.read_json(f),
    ".parquet": lambda f: pd.read_parquet(f),
}

def load_file(filepath: str) -> tuple:
    """Load any supported tabular file. Returns (df, status_message)."""
    ext = os.path.splitext(filepath)[1].lower()
    print(ext)
    if ext not in LOADERS:
        return None, f"❌ Unsupported type: `{ext}`. Supported: {', '.join(LOADERS)}"
    try:
        df = LOADERS[ext](filepath)
        # Auto-detect header row for Excel (first row with >40% non-null values)
        if ext in (".xlsx", ".xls"):
            for i, row in df.iterrows():
                if row.notna().mean() > 0.4:
                    df.columns = df.iloc[i]
                    df = df.iloc[i + 1:].reset_index(drop=True)
                    break
        df.columns = [str(c).strip() for c in df.columns]

        # make column names unique; duplicates cause `null_mask[col]` to be a
        # DataFrame which later leads to `TypeError: cannot convert the series
        # to <class 'int'>` when we cast stats values to int.
        if df.columns.duplicated().any():
            seen = {}
            new_cols = []
            for c in df.columns:
                if c in seen:
                    seen[c] += 1
                    new_cols.append(f"{c}.{seen[c]}")
                else:
                    seen[c] = 0
                    new_cols.append(c)
            df.columns = new_cols

        return df, f"✅ Loaded `{os.path.basename(filepath)}` — {len(df)} rows × {len(df.columns)} cols"
    except Exception as e:
        return None, f"❌ Load error: {e}"


### Gap Analyser

Returns a per-column breakdown of filled vs. missing cells and a boolean mask we reuse during filling.

In [6]:
def format_gap_report(stats: dict) -> str:
    lines = [
        f"**Total cells:** {stats['total']}  |  "
        f"**Filled:** {stats['filled']}  |  "
        f"**Missing:** {stats['missing']}  |  "
        f"**% filled:** {stats['pct']}%\n",
        "| Column | Filled | Missing | % Filled |",
        "|--------|--------|---------|----------|",
    ]
    for c in stats["col_stats"]:
        lines.append(f"| {c['column']} | {c['filled']} | {c['missing']} | {c['pct_filled']}% |")
    return "\n".join(lines)

def analyse_gaps(df: pd.DataFrame) -> dict:
    """Compute filled/missing statistics and a null mask."""

    def is_empty(x):
        if pd.isna(x): # treat pandas NA/NaN as empty
            return True
        return str(x).strip() in ("", "nan", "NaN", "None") # treat common "empty" strings as nulls

    # pandas >=2.1 renamed applymap to map; support both
    _mapper = df.map if hasattr(pd.DataFrame, "map") and "map" in dir(df) else df.applymap
    null_mask = _mapper(is_empty)

    filled = int((~null_mask).sum().sum()) # total non-empty cells
    total  = int(df.size) # total cells
    col_stats = []
    # iterate by position so we always get a Series even with duplicated names
    for idx, col in enumerate(df.columns):
        col_mask = null_mask.iloc[:, idx]
        n_null = int(col_mask.sum())
        n_filled = len(df) - n_null
        pct = round(100 * n_filled / len(df), 1) if len(df) else 0
        col_stats.append({
            "column": col,
            "filled": n_filled,
            "missing": n_null,
            "pct_filled": pct,
        })
    col_stats.sort(key=lambda x: x["pct_filled"])
    
    return {
        "total":     total,
        "filled":    filled,
        "missing":   total - filled,
        "pct":       round(100 * filled / total, 1) if total else 0,
        "col_stats": col_stats,
        "null_mask": null_mask,
    }


### Web Scraper

Fetches a user-supplied URL, strips navigation/script noise, and returns clean plain text for the LLM context.

In [7]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# def scrape_url(url: str, max_chars: int = 12_000) -> tuple:
#     """Scrape a URL and return (clean_text, status_message)."""
#     if not url.startswith(("http://", "https://")):
#         url = "https://" + url
#     try:
#         r = requests.get(url, headers=HEADERS, timeout=15)
#         r.raise_for_status()
#         soup = BeautifulSoup(r.text, "html.parser")
#         for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
#             tag.decompose()
#         text = soup.get_text(separator="\n", strip=True)
#         text = re.sub(r"\n{3,}", "\n\n", text)[:max_chars]
#         return text, f"✅ Scraped {url} — {len(text):,} chars extracted"
#     except Exception as e:
#         return "", f"❌ Scrape failed: {e}"

# Model name → their URL slug on artificialanalysis.ai
MODEL_URL_MAP = {
    "Claude Opus4.6(Adaptive)": "https://artificialanalysis.ai/models/claude-opus-4-5",
    "Grok 4":                   "https://artificialanalysis.ai/models/grok-4",
    "Claude 4.5Sonnet":         "https://artificialanalysis.ai/models/claude-sonnet-4-5",
    "Gemini 3 ProPreview(high)":"https://artificialanalysis.ai/models/gemini-2-5-pro",
    "Gemini 3Flash":            "https://artificialanalysis.ai/models/gemini-2-5-flash",
}

# Pages that have pricing/speed tables for all models
OVERVIEW_URLS = [
    "https://artificialanalysis.ai/leaderboards/models",
    "https://artificialanalysis.ai/models",
    "https://artificialanalysis.ai/price",
    "https://artificialanalysis.ai/models/claude-opus-4-5",
    "https://artificialanalysis.ai/models/claude-sonnet-4-5",
    "https://artificialanalysis.ai/models/grok-4",
    "https://artificialanalysis.ai/models/gemini-2-5-pro",
    "https://artificialanalysis.ai/models/gemini-2-5-flash",
]

def scrape_url(url: str, max_chars: int = 12_000) -> tuple:
    """Scrape a URL and return (clean_text, status_message)."""
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        text = re.sub(r"\n{3,}", "\n\n", text)[:max_chars]
        return text, f"✅ Scraped {url} — {len(text):,} chars extracted"
    except Exception as e:
        return "", f"❌ Scrape failed: {e}"

def scrape_all_context() -> str:
    """Scrape all overview pages and combine into one big context string."""
    all_text = []
    for url in OVERVIEW_URLS:
        text, status = scrape_url(url, max_chars=6000)
        print(status)
        if text:
            all_text.append(f"=== SOURCE: {url} ===\n{text}")
        time.sleep(1)  # be polite
    return "\n\n".join(all_text)

def scrape_with_js(url: str) -> str:
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page()
        page.goto(url, wait_until="networkidle")
        text = page.inner_text("body")
        browser.close()
        return text[:8000]

### LLM Prompt Builder

Each row gets its own prompt that includes:
- All existing (known) values for that row
- The list of columns that need filling
- The scraped web text as grounding context

In [8]:
SYSTEM_PROMPT = """
You are a precise data analyst. Your job is to fill in missing values in a table row.
You will be given the existing values for the row and a list of missing columns to fill.
Use the reference context provided to find accurate values.
Respond ONLY with a valid JSON object — no explanation, no markdown, no extra text.
Use "N/A" if a value genuinely cannot be determined.
"""

# def build_prompt(df: pd.DataFrame, null_mask: pd.DataFrame, context: str, row_idx: int) -> str:
#     row          = df.iloc[row_idx]
#     known        = {col: str(val) for col, val in row.items() if not null_mask.iloc[row_idx][col]}
#     missing_cols = [col for col in df.columns if null_mask.iloc[row_idx][col]]
#     example_out  = json.dumps({col: "value" for col in missing_cols})
#     return (
#         f"TABLE COLUMNS: {list(df.columns)}\n\n"
#         f"EXISTING VALUES FOR THIS ROW:\n{json.dumps(known, indent=2)}\n\n"
#         f"COLUMNS TO FILL: {missing_cols}\n\n"
#         f"REFERENCE CONTEXT (scraped from web):\n---\n{context[:8000]}\n---\n\n"
#         f"Return ONLY a JSON object like: {example_out}"
#     )
def build_prompt(df: pd.DataFrame, null_mask: pd.DataFrame, context: str, row_idx: int) -> str:
    row          = df.iloc[row_idx]
    known        = {col: str(val) for col, val in row.items() if not null_mask.iloc[row_idx][col]}
    missing_cols = [col for col in df.columns if null_mask.iloc[row_idx][col]]
    example_out  = json.dumps({col: "value" for col in missing_cols})
    
    # Try to get model-specific context
    model_name = str(row.get("MODEL", ""))
    model_context = ""
    if model_name in MODEL_URL_MAP:
        model_url = MODEL_URL_MAP[model_name]
        model_context, _ = scrape_url(model_url, max_chars=6000)
    
    combined_context = f"=== MODEL PAGE ===\n{model_context}\n\n=== GENERAL CONTEXT ===\n{context[:4000]}"
    
    return (
        f"TABLE COLUMNS: {list(df.columns)}\n\n"
        f"EXISTING VALUES FOR THIS ROW:\n{json.dumps(known, indent=2)}\n\n"
        f"COLUMNS TO FILL: {missing_cols}\n\n"
        f"You are filling data for model: {model_name}\n"
        f"Look for: pricing (input/output per 1M tokens), context window size, speed (tokens/sec), "
        f"latency (TTFT), parameters, knowledge cutoff date, license type.\n\n"
        f"REFERENCE CONTEXT (from artificialanalysis.ai):\n---\n{combined_context}\n---\n\n"
        f"Return ONLY a JSON object like: {example_out}"
    )

### LLM Caller

A single function that routes to OpenAI, Anthropic, or Ollama based on `provider`.
Each provider implements the same interface: send a prompt, get back text.

In [9]:
def call_llm(prompt: str, provider: str) -> str:
    """Call the chosen LLM and return raw text response."""
    if provider == "anthropic":
        resp = anthropic_client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.content[0].text

    elif provider == "ollama":
        resp = ollama_client.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": prompt},
            ],
        )
        return resp.choices[0].message.content

    else:  # openai (default)
        resp = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": prompt},
            ],
            max_tokens=1024,
        )
        return resp.choices[0].message.content

### Table Filler

Iterates over every row that has at least one missing cell, calls the LLM, parses the JSON response, and writes the values back into the dataframe.

In [10]:
def fill_table(df: pd.DataFrame, null_mask: pd.DataFrame, context: str, provider: str) -> pd.DataFrame:
    """Fill all missing cells row by row using the chosen LLM."""
    df_out = df.copy()
    rows_with_gaps = [i for i in range(len(df)) if null_mask.iloc[i].any()]

    for row_idx in tqdm(rows_with_gaps, desc="Filling rows"):
        prompt = build_prompt(df, null_mask, context, row_idx)
        try:
            raw = call_llm(prompt, provider)
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                fills = json.loads(match.group())
                for col, val in fills.items():
                    if col in df_out.columns:
                        df_out.at[row_idx, col] = val
        except Exception as e:
            print(f"  ⚠️  Row {row_idx} skipped: {e}")
            time.sleep(1)

    return df_out

## Time to put this together!

Quick smoke-test before launching the UI — verify each piece works independently.

In [11]:
# Test: load a file
# df, msg = load_file("LLM_MODEL_FITS.xlsx")
# print(msg)
# df.head()

In [12]:
# Test: analyse gaps
# stats = analyse_gaps(df)
# print(format_gap_report(stats))

In [13]:
# Test: scrape a URL
# text, status = scrape_url("https://artificialanalysis.ai")
# print(status)
# print(text[:500])

In [14]:
# Test: call the LLM directly
# print(call_llm("Reply with JSON: {\"test\": \"ok\"}", provider="openai"))

## Gradio UI

Four tabs mirror the pipeline steps:
1. Upload & Analyse
2. Web Scraper
3. LLM Auto-Fill
4. Edit & Export

In [15]:
# Shared state across tabs
state = {"df": None, "null_mask": None, "scraped_text": "", "df_filled": None}

In [16]:
with gr.Blocks(title="🧠 Smart Table Filler", theme=gr.themes.Soft(primary_hue="violet")) as demo:

    gr.Markdown("# 🧠 Smart Table Filler\nUpload a tabular file → analyse gaps → scrape a reference site → auto-fill with an LLM.")

    with gr.Tabs():

        # ── Tab 1: Upload & Analyse ────────────────────────────────────────────
        with gr.Tab("📂 1. Upload & Analyse"):
            file_input   = gr.File(label="Upload CSV, Excel, TSV, JSON, or Parquet",
                                   file_types=SUPPORTED_EXT)
            load_btn     = gr.Button("🔍 Load & Analyse", variant="primary")
            load_status  = gr.Markdown()
            gap_report   = gr.Markdown()
            preview      = gr.Dataframe(label="Preview (first 20 rows)", interactive=False)

            def on_load(file):
                if file is None:
                    return "Please upload a file.", "", None
                df, msg = load_file(file.name)
                if df is None:
                    return msg, "", None
                state["df"] = df
                stats = analyse_gaps(df)
                state["null_mask"] = stats["null_mask"]
                return msg, format_gap_report(stats), df.head(20)

            load_btn.click(on_load, inputs=file_input, outputs=[load_status, gap_report, preview])

        # ── Tab 2: Web Scraper ─────────────────────────────────────────────────
        with gr.Tab("🌐 2. Web Scraper"):
            gr.Markdown("Enter a URL to scrape as reference context for the LLM.")
            url_input      = gr.Textbox(label="Target URL",
                                        placeholder="e.g. https://artificialanalysis.ai")
            scrape_btn     = gr.Button("🕷️ Scrape", variant="primary")
            scrape_status  = gr.Markdown()
            scraped_text   = gr.Textbox(label="Scraped Text Preview (first 3 000 chars)",
                                        lines=15, interactive=False)

            def on_scrape(url):
                text = scrape_all_context()
                state["scraped_text"] = text
                return f"✅ Scraped {len(OVERVIEW_URLS)} pages — {len(text):,} total chars", text[:3000]
            # def on_scrape(url):
            #     if not url.strip():
            #         return "Please enter a URL.", ""
            #     text, msg = scrape_url(url.strip())
            #     state["scraped_text"] = text
            #     return msg, text[:3000]

            scrape_btn.click(on_scrape, inputs=url_input, outputs=[scrape_status, scraped_text])

        # ── Tab 3: LLM Auto-Fill ───────────────────────────────────────────────
        with gr.Tab("🤖 3. LLM Auto-Fill"):
            gr.Markdown("Choose a provider and fill all missing cells.")
            provider_dd  = gr.Dropdown(
                choices=["openai", "anthropic", "ollama"],
                value=DEFAULT_PROVIDER,
                label="LLM Provider",
            )
            fill_btn     = gr.Button("✨ Fill Missing Cells", variant="primary", size="lg")
            fill_status  = gr.Markdown()
            filled_table = gr.Dataframe(label="Filled Table", interactive=False)
            dl_file      = gr.File(label="⬇️ Download Filled Excel", visible=False)

            # This function is the core of the app: it checks prerequisites, calls the fill_table function, and handles the output.
            def on_fill(provider):
                if state["df"] is None:
                    return "⚠️ Load a file first (Tab 1).", None, gr.update(visible=False)
                if not state["scraped_text"]:
                    return "⚠️ No scraped text — run the scraper first (Tab 2).", None, gr.update(visible=False)
                try:
                    df_filled = fill_table(
                        state["df"], state["null_mask"], state["scraped_text"], provider
                    )
                    state["df_filled"] = df_filled
                    out_path = "C:\\Users\\Nii Quaynor\\projects\\Caishen\\tmp\\filled_table.xlsx"
                    df_filled.to_excel(out_path, index=False)
                    before = state["null_mask"].sum().sum()
                    after  = analyse_gaps(df_filled)["missing"]
                    msg = (
                        f"✅ **Done!** Cells filled: {before - after}  |  "
                        f"Still missing: {after}"
                    )
                    return msg, df_filled, gr.update(value=out_path, visible=True)
                except Exception as e:
                    return f"❌ Error: {e}", None, gr.update(visible=False) # hide download if error

            fill_btn.click(on_fill, inputs=provider_dd,
                           outputs=[fill_status, filled_table, dl_file])

        # ── Tab 4: Edit & Export ───────────────────────────────────────────────
        with gr.Tab("✏️ 4. Edit & Export"):
            gr.Markdown("Review and manually edit the filled table, then export.")
            edit_table   = gr.Dataframe(label="Edit Table", interactive=True)
            refresh_btn  = gr.Button("🔄 Load Filled Table")
            with gr.Row():
                csv_btn  = gr.Button("📥 Export CSV")
                xlsx_btn = gr.Button("📥 Export Excel")
            export_status = gr.Markdown()
            export_file   = gr.File(label="Download", visible=False)

            def on_refresh():
                df = state.get("df_filled") or state.get("df")
                return df

            def on_export_csv(data):
                path = "C:\\Users\\Nii Quaynor\\projects\\Caishen\\tmp\\export.csv"
                pd.DataFrame(data).to_csv(path, index=False)
                return "✅ CSV ready!", gr.update(value=path, visible=True)

            def on_export_xlsx(data):
                path = "C:\\Users\\Nii Quaynor\\projects\\Caishen\\tmp\\export.xlsx"
                pd.DataFrame(data).to_excel(path, index=False)
                return "✅ Excel ready!", gr.update(value=path, visible=True)

            refresh_btn.click(on_refresh, outputs=edit_table)
            csv_btn.click(on_export_csv,  inputs=edit_table, outputs=[export_status, export_file])
            xlsx_btn.click(on_export_xlsx, inputs=edit_table, outputs=[export_status, export_file])

## What could possibly come next? 😂

In [17]:
demo.launch(share=True, allowed_paths=["C:\\Users\\Nii Quaynor\\projects\\Caishen\\tmp"])

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://507b6440ec4f5e7ea7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Admit it — you thought table auto-filling would be more complicated than that!!